# Setup environment and install requirements

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [2]:
warnings.filterwarnings("ignore")

# Data loader

In [3]:
df = pd.read_csv("../data/Ridership.csv")
df.head(10)

,Year,Month,Day,Week Number,Corridor,Workday,Station,Period,Ridership,N_trains,Covid19
0,2019,January,1,1,Corridor_1,n,Station_1,Weekend/Holiday,174,3,0
1,2019,January,1,1,Corridor_1,n,Station_2,Weekend/Holiday,378,10,0
2,2019,January,1,1,Corridor_1,n,Station_3,Weekend/Holiday,599,12,0
3,2019,January,1,1,Corridor_2,n,Station_4,Weekend/Holiday,2759,35,0
4,2019,January,1,1,Corridor_2,n,Station_5,Weekend/Holiday,2629,36,0
5,2019,January,1,1,Corridor_2,n,Station_3,Weekend/Holiday,27,1,0
6,2019,January,1,1,Corridor_3,n,Station_4,Weekend/Holiday,3321,35,0
7,2019,January,1,1,Corridor_3,n,Station_5,Weekend/Holiday,3721,36,0
8,2019,January,1,1,Corridor_3,n,Station_3,Weekend/Holiday,17,1,0
9,2019,January,2,1,Corridor_1,y,Station_1,AM Peak,3519,6,0


In [4]:
df["date"] = pd.to_datetime(df["Year"].astype("str") + "-" + df["Month"] + "-" + df["Day"].astype("str"))


In [5]:
station_counts = df['Station'].value_counts()
valid_stations = station_counts[station_counts >= 30].index

df_main = df[df['Station'].isin(valid_stations)]
df_rare = df[~df['Station'].isin(valid_stations)]

df_rare['Station'] = 'Other'

df = pd.concat([df_main,df_rare])
df = df.sort_values("date").reset_index().drop("index",axis=1)
df

,Year,Month,Day,Week Number,Corridor,Workday,Station,Period,Ridership,N_trains,Covid19,date
0,2019,January,1,1,Corridor_1,n,Station_1,Weekend/Holiday,174,3,0,2019-01-01
1,2019,January,1,1,Corridor_3,n,Station_5,Weekend/Holiday,3721,36,0,2019-01-01
2,2019,January,1,1,Corridor_3,n,Station_4,Weekend/Holiday,3321,35,0,2019-01-01
3,2019,January,1,1,Corridor_2,n,Station_3,Weekend/Holiday,27,1,0,2019-01-01
4,2019,January,1,1,Corridor_3,n,Station_3,Weekend/Holiday,17,1,0,2019-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...
64364,2022,December,31,52,Corridor_7,n,Station_24,Weekend/Holiday,686,12,0,2022-12-31
64365,2022,December,31,52,Corridor_7,n,Station_21,Weekend/Holiday,82,3,0,2022-12-31
64366,2022,December,31,52,Corridor_7,n,Station_3,Weekend/Holiday,419,15,0,2022-12-31
64367,2022,December,31,52,Corridor_2,n,Station_43,Weekend/Holiday,3053,36,0,2022-12-31


In [6]:
def time_split_by_station(df, split_date):
    train_list = []
    test_list  = []

    for station, g in df.groupby('Station'):
        g = g.sort_values('date')
        train_list.append(g[g['date'] < split_date])
        test_list.append(g[g['date'] >= split_date])

    return pd.concat(train_list), pd.concat(test_list)


In [7]:
def add_calendar_features(df):
    df = df.copy()

    df['month'] = df['date'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    return df


In [8]:
def create_time_series_features(df):
    df = df.copy()
    df = df.sort_values(by=['Station', 'date'])

    for lag in [1, 2, 3]:
        df[f'lag_{lag}_period'] = (
            df.groupby(['Station', 'Period'])['Ridership']
              .shift(lag)
        )

    df['rolling_mean_3_period'] = (
        df.groupby(['Station', 'Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

    df['rolling_mean_7_period'] = (
        df.groupby(['Station', 'Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    )

    return df


In [9]:
def target_encode_station(train_df, test_df, target_col='Ridership'):
    """
    Time-safe target encoding for Station.
    Fit on train only, apply to test.
    """
    station_mean = train_df.groupby('Station')[target_col].mean()
    global_mean = train_df[target_col].mean()

    train_df = train_df.copy()
    test_df  = test_df.copy()

    train_df['Station_enc'] = train_df['Station'].map(station_mean)
    test_df['Station_enc']  = test_df['Station'].map(station_mean)

    # unseen / rare stations in test
    test_df['Station_enc'] = test_df['Station_enc'].fillna(global_mean)

    return train_df, test_df


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_preprocessor():

    categorical_features = ['Corridor', 'Workday', 'Period']
    
    numeric_features = [
        'Station_enc',
        'month_sin', 'month_cos',
        'day_of_week', 'is_weekend',
        'Covid19',
        'lag_1_period', 'lag_2_period', 'lag_3_period',
        'rolling_mean_3_period', 'rolling_mean_7_period'
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', StandardScaler(), numeric_features)
        ],
        remainder='drop'
    )

    return preprocessor


In [ ]:
# === After safe preprocessing ===
df = add_calendar_features(df)

# === Time-based split (performed separately within each station) ===
train_df, test_df = time_split_by_station(df, split_date='2022-01-01')

# === Outlier Removal ===
# Compute IQR bounds using TRAIN data only (to avoid data leakage)
Q1 = train_df["Ridership"].quantile(0.25)
Q3 = train_df["Ridership"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Compute median value from TRAIN set
median_value = train_df["Ridership"].median()

# Replace outliers in TRAIN and TEST using TRAIN-derived thresholds
train_df.loc[
    ((train_df["Ridership"] < lower_bound) | 
     (train_df["Ridership"] > upper_bound)),
    "Ridership"
] = median_value

test_df.loc[
    ((test_df["Ridership"] < lower_bound) | 
     (test_df["Ridership"] > upper_bound)),
    "Ridership"
] = median_value


# === Create lag and rolling features (only after split to prevent leakage) ===
train_df = create_time_series_features(train_df)
test_df  = create_time_series_features(test_df)

# Drop rows with NaN values introduced by lag/rolling operations
train_df = train_df.dropna()
test_df  = test_df.dropna()

# === Apply target encoding for Station (fit on TRAIN, apply to TEST) ===
train_df, test_df = target_encode_station(train_df, test_df)

# Optional: inspect encoded station values
stations = train_df.groupby(['Station', 'Station_enc']).size()

# === Separate features (X) and target (y) ===
X_train = train_df.drop(columns=['Ridership', 'Station', 'date'])
y_train = train_df['Ridership']

X_test  = test_df.drop(columns=['Ridership', 'Station', 'date'])
y_test  = test_df['Ridership']

# === Encoding and scaling ===
# Fit preprocessing pipeline on TRAIN only
preprocessor = build_preprocessor()

X_train_enc = preprocessor.fit_transform(X_train)

# Apply the same transformations to TEST
X_test_enc  = preprocessor.transform(X_test)


In [12]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score


model = ElasticNet(max_iter=5000)

param_grid = {
    'alpha': np.logspace(-3, 2, 6),
    'l1_ratio': [0.1, 0.5, 0.9]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    error_score='raise'  
)

grid_search.fit(X_train_enc, y_train)

best_model = grid_search.best_estimator_

print("Best alpha:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)


from sklearn.metrics import mean_squared_error, r2_score


y_pred = best_model.predict(X_test_enc)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R2  : {r2:.3f}")



Best alpha: {'alpha': np.float64(0.001), 'l1_ratio': 0.9}
Best CV RMSE: 314.6638295997833
Test RMSE: 435.30
Test R2  : 0.480


In [13]:

results_df = test_df.copy()
results_df['predicted_ridership'] = y_pred


def allocate_trains(df, capacity=600, min_trains=1):
    df = df.copy()
    
    df['required_trains'] = np.ceil(
        df['predicted_ridership'] / capacity
    ).astype(int)
    
    # حداقل یک قطار (اختیاری ولی منطقی)
    df['required_trains'] = df['required_trains'].clip(lower=min_trains)
    
    return df

capacity = 600

results_df = allocate_trains(
    results_df,
    capacity=capacity,
    min_trains=1
)

allocation_summary = (
    results_df
    .groupby(['Station', 'date', 'Period'], as_index=False)
    .agg({
        'predicted_ridership': 'sum',
        'required_trains': 'max'
    })
)




In [14]:
results_df[["Ridership", "predicted_ridership"]]

,Ridership,predicted_ridership
55359,161,237.912828
55966,54,130.671763
56303,124,242.184666
56294,271,280.835210
56624,24,236.462299
...,...,...
64271,133,116.420284
64309,143,122.442884
64317,478,418.788130
64308,296,249.978994


In [15]:
import joblib

model = joblib.dump(model, f'Model.pkl')